In [28]:
!pip install langchian chromadb openai tiktoken pypdf langchain_openai langchain-community

In [29]:
!pip install langchain-chroma
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [30]:
from langchain_core.documents import Document

# Create langchain documents for Pakistani players

doc1 = Document(
    page_content=(
        "Babar Azam is a Pakistani right-handed batter from Lahore. "
        "He is known for his elegant cover drive and consistent run-scoring "
        "across Test, ODI and T20 cricket."
    ),
    metadata={"name": "Babar Azam", "role": "Batter", "city": "Lahore"},
)

doc2 = Document(
    page_content=(
        "Shaheen Shah Afridi is a left-arm fast bowler. "
        "He is known for swinging the new ball and taking early wickets."
    ),
    metadata={"name": "Shaheen Shah Afridi", "role": "Fast bowler", "city": "Landi Kotal"},
)

doc3 = Document(
    page_content=(
        "Mohammad Rizwan is a wicketkeeper-batter from Peshawar. "
        "He is known for his fitness, his composure under pressure and "
        "his reliable batting at the top of the order."
    ),
    metadata={"name": "Mohammad Rizwan", "role": "Wicketkeeper-batter", "city": "Peshawar"},
)

doc4 = Document(
    page_content=(
        "Shadab Khan is a leg-spin all-rounder. "
        "He contributes with the ball in the middle overs, with quick "
        "runs lower down the order and with sharp fielding."
    ),
    metadata={"name": "Shadab Khan", "role": "All-rounder", "city": "Mianwali"},
)

doc5 = Document(
    page_content=(
        "Haris Rauf is a right-arm fast bowler from Rawalpindi. "
        "He is known for his pace and for bowling in the powerplay and "
        "at the death in T20 cricket."
    ),
    metadata={"name": "Haris Rauf", "role": "Fast bowler", "city": "Rawalpindi"},
)



In [31]:
# free hugging face embeddings

!pip install -q langchain-huggingface sentence-transformers langchain-chroma

In [32]:
from langchain_huggingface import HuggingFaceEmbeddings

In [33]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory="chroma_db",
    collection_name="players_v2",      # new name, so no old duplicates
)

docs = [doc1, doc2, doc3, doc4, doc5]
vector_store.add_documents(docs, ids=["babar", "shaheen", "rizwan", "shadab", "rauf"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

['babar', 'shaheen', 'rizwan', 'shadab', 'rauf']

In [34]:
# add documents
vector_store.add_documents(docs)    # create unique id for document object

['5f71a903-c315-4ef6-a2c7-d115202cc3a5',
 '9f31cdd5-f3aa-4852-8416-e7aca0a0cba7',
 'c0a1ff6d-f845-4263-b4a2-2859efe9de2e',
 '2a653c50-8418-4762-9fed-3f354ec0db93',
 'dc42b68b-5cbd-46b8-88af-974324fc66bf']

In [35]:
# fetching verctors(view documents)

vector_store.get(include=['documents', 'embeddings', 'metadatas'])

{'ids': ['babar',
  'shaheen',
  'rizwan',
  'shadab',
  'rauf',
  '5f71a903-c315-4ef6-a2c7-d115202cc3a5',
  '9f31cdd5-f3aa-4852-8416-e7aca0a0cba7',
  'c0a1ff6d-f845-4263-b4a2-2859efe9de2e',
  '2a653c50-8418-4762-9fed-3f354ec0db93',
  'dc42b68b-5cbd-46b8-88af-974324fc66bf'],
 'embeddings': array([[-0.02696148,  0.0637865 , -0.10695163, ..., -0.06333047,
         -0.03499779,  0.03784728],
        [ 0.00958232,  0.04848748, -0.01807215, ..., -0.01617979,
          0.03939397,  0.00408933],
        [-0.05557184,  0.10867379, -0.04831053, ..., -0.05811072,
          0.04100439,  0.04581537],
        ...,
        [-0.05557184,  0.10867379, -0.04831053, ..., -0.05811072,
          0.04100439,  0.04581537],
        [ 0.04702629,  0.04941409, -0.08942741, ..., -0.03383321,
          0.03149263, -0.02480498],
        [ 0.01414308,  0.00958196, -0.06570488, ...,  0.01703917,
          0.02897388,  0.01771654]], shape=(10, 384)),
 'documents': ['Babar Azam is a Pakistani right-handed batter from

In [36]:
# Searching documents

vector_store.similarity_search(
                                query = " Who is the best player among the team with respect to bowling",
                                k = 2
                              )

[Document(id='shaheen', metadata={'name': 'Shaheen Shah Afridi', 'city': 'Landi Kotal', 'role': 'Fast bowler'}, page_content='Shaheen Shah Afridi is a left-arm fast bowler. He is known for swinging the new ball and taking early wickets.'),
 Document(id='9f31cdd5-f3aa-4852-8416-e7aca0a0cba7', metadata={'city': 'Landi Kotal', 'role': 'Fast bowler', 'name': 'Shaheen Shah Afridi'}, page_content='Shaheen Shah Afridi is a left-arm fast bowler. He is known for swinging the new ball and taking early wickets.')]

In [37]:
# Searching documents   and filter the documents on the basis of metadata like here role and gives the similarity score(less more better)

vector_store.similarity_search_with_score(
                                query = "",
                                filter = {"role":"Fast bowler"}
                                           )

[(Document(id='shaheen', metadata={'name': 'Shaheen Shah Afridi', 'city': 'Landi Kotal', 'role': 'Fast bowler'}, page_content='Shaheen Shah Afridi is a left-arm fast bowler. He is known for swinging the new ball and taking early wickets.'),
  1.8324040174484253),
 (Document(id='9f31cdd5-f3aa-4852-8416-e7aca0a0cba7', metadata={'role': 'Fast bowler', 'name': 'Shaheen Shah Afridi', 'city': 'Landi Kotal'}, page_content='Shaheen Shah Afridi is a left-arm fast bowler. He is known for swinging the new ball and taking early wickets.'),
  1.8324040174484253),
 (Document(id='rauf', metadata={'city': 'Rawalpindi', 'name': 'Haris Rauf', 'role': 'Fast bowler'}, page_content='Haris Rauf is a right-arm fast bowler from Rawalpindi. He is known for his pace and for bowling in the powerplay and at the death in T20 cricket.'),
  1.9014286994934082),
 (Document(id='dc42b68b-5cbd-46b8-88af-974324fc66bf', metadata={'city': 'Rawalpindi', 'name': 'Haris Rauf', 'role': 'Fast bowler'}, page_content='Haris Rauf 

In [42]:
# udpate documents

updated_doc2 = Document(
    page_content = "Naseem Shah is a right-arm fast bowler from Lower Dir. "
        "He is known for his pace and for taking wickets with the new ball "
        "at a young age.",
    metadata = {"name": "Naseem Shah", "role": "Fast bowler", "city": "Lower Dir"},
)

vector_store.update_document(document_id="0dc0432c-9754-4da5-a19a-6466bf2365ec", document = updated_doc2)

print(vector_store.get(ids=["shaheen"]))

for r in vector_store.similarity_search("Naseem Shah", k=2):
    print(r.metadata["name"])

{'ids': ['shaheen'], 'embeddings': None, 'documents': ['Shaheen Shah Afridi is a left-arm fast bowler. He is known for swinging the new ball and taking early wickets.'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'name': 'Shaheen Shah Afridi', 'role': 'Fast bowler', 'city': 'Landi Kotal'}]}
Shaheen Shah Afridi
Shaheen Shah Afridi


In [39]:
print(docs)
print(vector_store)

[Document(metadata={'name': 'Babar Azam', 'role': 'Batter', 'city': 'Lahore'}, page_content='Babar Azam is a Pakistani right-handed batter from Lahore. He is known for his elegant cover drive and consistent run-scoring across Test, ODI and T20 cricket.'), Document(metadata={'name': 'Shaheen Shah Afridi', 'role': 'Fast bowler', 'city': 'Landi Kotal'}, page_content='Shaheen Shah Afridi is a left-arm fast bowler. He is known for swinging the new ball and taking early wickets.'), Document(metadata={'name': 'Mohammad Rizwan', 'role': 'Wicketkeeper-batter', 'city': 'Peshawar'}, page_content='Mohammad Rizwan is a wicketkeeper-batter from Peshawar. He is known for his fitness, his composure under pressure and his reliable batting at the top of the order.'), Document(metadata={'name': 'Shadab Khan', 'role': 'All-rounder', 'city': 'Mianwali'}, page_content='Shadab Khan is a leg-spin all-rounder. He contributes with the ball in the middle overs, with quick runs lower down the order and with sharp

In [41]:
# data = vector_store.get()
for id_, meta in zip(data["ids"], data["metadatas"]):
    print(id_, "|", meta["name"])

babar | Babar Azam
shaheen | Shaheen Shah Afridi
rizwan | Mohammad Rizwan
shadab | Shadab Khan
rauf | Haris Rauf
5f71a903-c315-4ef6-a2c7-d115202cc3a5 | Babar Azam
9f31cdd5-f3aa-4852-8416-e7aca0a0cba7 | Shaheen Shah Afridi
c0a1ff6d-f845-4263-b4a2-2859efe9de2e | Mohammad Rizwan
2a653c50-8418-4762-9fed-3f354ec0db93 | Shadab Khan
dc42b68b-5cbd-46b8-88af-974324fc66bf | Haris Rauf
